# 2 · `_unnormalized`

**Trusts:** notebook 1 (the factorization).

`_unnormalized(row)` is the core of the class: inference, the E-step and the log-likelihood are all built on it. It returns `(hidden, table)`:

- `hidden`: the modeled nodes this row doesn't observe (NaN, or no column at all)
- `table[i, j, …]` = P(hidden = (i, j, …), observed cells | inputs): the product from notebook 1, one entry per way of filling in the hidden nodes

We build it in four steps: **no hidden → loop over nodes → one hidden → any number hidden.**

In [ ]:
%load_ext autoreload
%autoreload 2

import itertools

import numpy as np
import pandas as pd
from toy import (
    CARDS,
    CO,
    DATA,
    EVIDENCE,
    NO2,
    S1,
    S4,
    STRUCTURE,
    TABLES,
    H,
    T,
    cpts,
    network,
    same,
    toy_bn,
)

from cse250a.modules.core import CPT

net = network()
bn = toy_bn()   # DiscreteBN with the hand-picked CPTs, not fitted

## Step 1: a complete row, written out

Row 0 is fully observed, so there's nothing to fill in: the "table" is one number, the product of four CPT entries.

In [ ]:
row = DATA.iloc[0]
h, t, co, no2, s1, s4 = (int(row[c]) for c in (H, T, CO, NO2, S1, S4))

step1 = TABLES[CO][h, co] * TABLES[NO2][h, no2] * TABLES[S1][co, t, s1] * TABLES[S4][no2, co, s4]
print(row.to_dict(), "\n→", step1)

## Step 2: loop over the nodes

Writing the product by hand doesn't scale. Every factor follows the same rule: **look up the node's CPT at (its parents' values, its own value)**. `net.family(node)` gives exactly those columns, in the CPT's axis order.

In [ ]:
def product(values):
    '''values: {column: bin} for every node and input. Returns Π CPT entries.'''
    p = 1.0
    for node in net.structure:
        index = tuple(int(values[c]) for c in net.family(node))
        p *= bn.cpts[node].table[index]
    return p

step2 = product(row)
same("loop over nodes = written out", step2, step1)

## Step 3: one hidden node

Row 2 is missing CO. We don't know CO, so we compute the product **for each value CO could take**. That gives a vector with one entry per CO state:

$$\text{table}[c] = P(\text{CO}=c,\ \text{the observed cells} \mid \text{inputs})$$

In [ ]:
row = DATA.iloc[2]
print(row.to_dict())

step3 = np.zeros(CARDS[CO])
for c in range(CARDS[CO]):
    filled = {**row.to_dict(), CO: c}   # fill in the hidden node
    step3[c] = product(filled)
print("table over CO:", step3.round(5))

Two things to notice:

- `step3.sum()` = P(observed cells | inputs). Summing over CO's states *marginalizes CO out*. Notebook 3 uses this as the row's likelihood.
- `step3 / step3.sum()` = P(CO | observed): the posterior. Notebook 3 again.

## Step 4: any number of hidden nodes

Same idea with more than one hidden node: **one table axis per hidden node**, and fill every combination. `itertools.product` walks every combination of states:

In [ ]:
print(list(itertools.product(range(2), range(2))))   # every (CO, NO2) pair for 2 hidden binary nodes

In [ ]:
def unnormalized(row):
    hidden = [n for n in net.structure if pd.isna(row[n])]
    table = np.zeros(tuple(CARDS[h] for h in hidden))
    for states in itertools.product(*[range(CARDS[h]) for h in hidden]):
        filled = {**row.to_dict(), **dict(zip(hidden, states))}
        table[states] = product(filled)
    return hidden, table


for i in (0, 2, 4, 7):
    hidden, table = unnormalized(DATA.iloc[i])
    print(f"row {i}: hidden {[str(h) for h in hidden]!s:42} table shape {table.shape}")

Row 0 has nothing hidden, so `table` has shape `()`: a 0-d array holding one number. `itertools.product()` with no ranges yields exactly one empty tuple, so the loop runs once and `table[()]` is set. That's step 1 again, without a special case.

**Compare with the class on every row:**

In [ ]:
for i, row in DATA.iterrows():
    mine_h, mine_t = unnormalized(row)
    their_h, their_t = bn._unnormalized(row)
    assert mine_h == their_h
    same(f"row {i}  hidden={[str(h) for h in mine_h]}", mine_t, their_t)

## Step 5: latent nodes are just "hidden on every row"

A latent node has **no column at all**. The class's hidden test is `n not in row.index or pd.isna(row[n])`, so a node with no column counts as hidden, the same as a NaN. Drop CO's column from row 2 and nothing changes:

In [ ]:
row = DATA.iloc[2]
same("CO as NaN  =  no CO column at all",
     bn._unnormalized(row)[1], bn._unnormalized(row.drop(CO))[1])

## A sanity check from the definition

For row 4 (CO and NO2 hidden), `table.sum()` = P(S1=1, S4=1 | hour=1, T=0). Do it for **every** possible (S1, S4) observation: the four probabilities must add up to 1, because some observation always happens.

In [ ]:
total = 0.0
for s1, s4 in itertools.product(range(2), repeat=2):
    r = DATA.iloc[4].copy()
    r[S1], r[S4] = s1, s4
    p = bn._unnormalized(r)[1].sum()
    total += p
    print(f"P(S1={s1}, S4={s4} | hour=1, T=0) = {p:.4f}")
print("sum:", round(total, 12))

## The final method

Here is `_unnormalized` from `bn.py`. Every line maps to a step above:

| ours | `bn.py` |
|---|---|
| `hidden = [n ... if pd.isna(row[n])]` | same, plus `n not in row.index` for latent nodes (step 5) |
| `itertools.product(...)` over hidden states | same |
| `product(filled)`: loop over nodes, look up `family` | inlined: `assignment[var] if var in hidden else int(row[var])` |

In [ ]:
import inspect

from cse250a.modules.models import DiscreteBN

print(inspect.getsource(DiscreteBN._unnormalized))